In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from google.colab import files

u = files.upload()

Saving housing.csv to housing.csv


In [ ]:
data = pd.read_csv("housing.csv")

In [ ]:
pd.isnull(data).sum()

,0
longitude,0
latitude,0
housing_median_age,0
total_rooms,0
total_bedrooms,207
population,0
households,0
median_income,0
median_house_value,0
ocean_proximity,0


In [ ]:
data = data.fillna(0)

pd.isnull(data).sum()

,0
longitude,0
latitude,0
housing_median_age,0
total_rooms,0
total_bedrooms,0
population,0
households,0
median_income,0
median_house_value,0
ocean_proximity,0


In [ ]:

data = data.drop('ocean_proximity', axis=1)
X = data.drop('median_house_value', axis=1).values
y = data['median_house_value'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train).T
X_test = scaler.transform(X_test).T

y_train = y_train.reshape(-1, 1) / 1e6
y_test = y_test.reshape(-1, 1) / 1e6

num_features = X_train.shape[0]
layer_sizes = [num_features, 64, 32, 1]

def relu(z):
    return np.maximum(0, z)

def relu_derivative(z):
    return z > 0

def mse_loss(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)

def mse_loss_derivative(y_true, y_pred):
    m = y_true.shape[0]
    return (y_pred - y_true) / m

def init_params(layer_sizes):
    params = {}
    for i in range(1, len(layer_sizes)):
        if i == len(layer_sizes) - 1:
            params['W' + str(i)] = np.random.randn(layer_sizes[i], layer_sizes[i-1]) * np.sqrt(1.0 / layer_sizes[i-1])
        else:
            params['W' + str(i)] = np.random.randn(layer_sizes[i], layer_sizes[i-1]) * np.sqrt(2.0 / layer_sizes[i-1])
        params['b' + str(i)] = np.zeros((layer_sizes[i], 1))
    return params

def forward_propagation(X, params, layer_sizes):
    cache = {'A0': X}
    A = X

    for i in range(1, len(layer_sizes) - 1):
        Z = np.dot(params['W' + str(i)], A) + params['b' + str(i)]
        A = relu(Z)
        cache['Z' + str(i)] = Z
        cache['A' + str(i)] = A

    Z_final = np.dot(params['W' + str(len(layer_sizes) - 1)], A) + params['b' + str(len(layer_sizes) - 1)]
    A_final = Z_final
    cache['Z' + str(len(layer_sizes) - 1)] = Z_final
    cache['A' + str(len(layer_sizes) - 1)] = A_final

    return A_final, cache

def backward_propagation(y_true, params, cache, layer_sizes, clip_value=5.0):
    grads = {}
    m = y_true.shape[0]
    L = len(layer_sizes) - 1

    dZ = mse_loss_derivative(y_true.T, cache['A' + str(L)])
    grads['dW' + str(L)] = np.dot(dZ, cache['A' + str(L-1)].T) / m
    grads['db' + str(L)] = np.sum(dZ, axis=1, keepdims=True) / m

    grads['dW' + str(L)] = np.clip(grads['dW' + str(L)], -clip_value, clip_value)
    grads['db' + str(L)] = np.clip(grads['db' + str(L)], -clip_value, clip_value)

    for i in range(L-1, 0, -1):
        dA = np.dot(params['W' + str(i+1)].T, dZ)
        dZ = dA * relu_derivative(cache['Z' + str(i)])
        grads['dW' + str(i)] = np.dot(dZ, cache['A' + str(i-1)].T) / m
        grads['db' + str(i)] = np.sum(dZ, axis=1, keepdims=True) / m

        grads['dW' + str(i)] = np.clip(grads['dW' + str(i)], -clip_value, clip_value)
        grads['db' + str(i)] = np.clip(grads['db' + str(i)], -clip_value, clip_value)

    return grads

def train(X, y, layer_sizes, learning_rate=0.0001, epochs=1000):

    params = init_params(layer_sizes)

    for epoch in range(epochs):

        y_pred, cache = forward_propagation(X, params, layer_sizes)


        loss = mse_loss(y.T, y_pred)

        grads = backward_propagation(y, params, cache, layer_sizes)

        for i in range(1, len(layer_sizes)):
            params['W' + str(i)] -= learning_rate * grads['dW' + str(i)]
            params['b' + str(i)] -= learning_rate * grads['db' + str(i)]

        if epoch % 100 == 0:
            print(f"Epoch {epoch}, Loss: {loss}")

    return params

def predict(X, params, layer_sizes):
    y_pred, _ = forward_propagation(X, params, layer_sizes)
    return y_pred * 1e6

params = train(X_train, y_train, layer_sizes, learning_rate=0.0001, epochs=1000)

predictions = predict(X_test, params, layer_sizes)
print("Predicted values on test set:", predictions)


Epoch 0, Loss: 0.24138852733662094
Epoch 100, Loss: 0.21952497065368795
Epoch 200, Loss: 0.20276531480832816
Epoch 300, Loss: 0.1894373445469557
Epoch 400, Loss: 0.17846940056141228
Epoch 500, Loss: 0.16918209373333032
Epoch 600, Loss: 0.16112515215800943
Epoch 700, Loss: 0.15400051436259823
Epoch 800, Loss: 0.1476050341860755
Epoch 900, Loss: 0.141800074287179
Predicted values on test set: [[ 258758.40236565 -114663.27108331   -3995.60567182 ... 1061110.11222746
     2983.13743535    4873.51773637]]


In [ ]:
from sklearn.metrics import mean_squared_error, r2_score

predictions = predictions.reshape(-1, 1)

mse = mean_squared_error(y_test, predictions / 1e6)
r2 = r2_score(y_test, predictions / 1e6)

print(f"Mean Squared Error on Test Set: {mse}")
print(f"R-squared on Test Set: {r2}")

Mean Squared Error on Test Set: 0.14133307581337062
R-squared on Test Set: -9.78541723692224
